# 04 — Mistral vs Mistral + RAG

**Objectif :** comparer deux approches de traduction médicale EN->FR :
- **Mistral** : traduction directe via `mistral-small-latest`
- **Mistral + RAG** : même modèle, mais le prompt inclut les 3 abstracts FR les plus
  similaires à la source, récupérés via embeddings denses + FAISS

**Métriques :** MEDCON-like F1 + BLEU




## 0. Setup

In [ ]:
import subprocess, os
subprocess.run(['pip', 'install', '-q', '--upgrade', 'numpy'])
os.kill(os.getpid(), 9)

In [1]:
import os

if not os.path.exists('/content/Evaluating_medical_machine_translation'):
    !git clone https://github.com/11Maroua/Evaluating_medical_machine_translation.git

os.chdir('/content/Evaluating_medical_machine_translation')
print(f'Répertoire : {os.getcwd()}')

Répertoire : /content/Evaluating_medical_machine_translation


In [2]:
!pip install -q pytorch-lightning==1.9.5 torchmetrics entmax jsonargparse==3.13.1
!pip install -q "transformers==4.40.0" --force-reinstall

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires numpy<2.0.0,>=1.20.0, but you have numpy 2.4.4 which is incompatible.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 5.29.6 which is incompatible.
unbabel-comet 2.2.7 requires pytorch-lightning<3.0.0,>=2.0.0, but you have pytorch-lightning 1.9.5 which is incompatible.
unbabel-comet 2.2.7 requires torchmetrics<0.11.0,>=0.10.2, but you have torchmetrics 1.9.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2

In [3]:
from google.colab import files

os.makedirs('data', exist_ok=True)

# Uploader : corpus_WMT24.json + unique_contexts_for_RAG.json + cleaned_mesh_snomed_dico.json
uploaded = files.upload()
for fname in uploaded:
    dest = f'data/{fname}'
    os.replace(fname, dest)
    print(f'  {dest}')

Saving cleaned_mesh_snomed_dico.json to cleaned_mesh_snomed_dico.json
Saving corpus_WMT24.json to corpus_WMT24.json
Saving unique_contexts_for_RAG.json to unique_contexts_for_RAG.json
  data/cleaned_mesh_snomed_dico.json
  data/corpus_WMT24.json
  data/unique_contexts_for_RAG.json


In [4]:
!pip uninstall mistralai -y
!pip install mistralai==1.2.5 --quiet
!pip install -q pyahocorasick sacrebleu sentence-transformers faiss-cpu transformers accelerate

Found existing installation: mistralai 1.2.5
Uninstalling mistralai-1.2.5:
  Successfully uninstalled mistralai-1.2.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 1.13.0 which is incompatible.
unbabel-comet 2.2.7 requires numpy<2.0.0,>=1.20.0, but you have numpy 2.4.4 which is incompatible.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 5.29.6 which is incompatible.
unbabel-comet 2.2.7 requires pytorch-lightning<3.0.0,>=2.0.0, but you have pytorch-lightning 1.9.5 which is incompatible.
unbabel-comet 2.2.7 requires torchmetrics<0.11.0,>=0.10.2, but you have torchmetrics 1.9.0 which is incompatible.
unbabel-comet 2.2.7 requires transformers<5.0,>=4.17, but you have transformers 5.7.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2

## 1. Imports

In [5]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import torch
import sacrebleu
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import faiss
from mistralai import Mistral
from google.colab import userdata

mistral_client = Mistral(api_key=userdata.get('MISTRAL_API_KEY'))
print('Mistral OK')

os.chdir('/content/Evaluating_medical_machine_translation')
sys.path.insert(0, 'src')
from medcon import build_pairs, build_automaton, medcon_grouped

print('Imports OK')

Mistral OK
Imports OK


## 2. Chargement des données

In [6]:
with open('data/corpus_WMT24.json', encoding='utf-8') as f:
    test_docs = json.load(f)

with open('data/unique_contexts_for_RAG.json', encoding='utf-8') as f:
    rag_contexts = json.load(f)

with open('data/cleaned_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict = json.load(f)

rag_texts = [str(c) for c in rag_contexts]

print(f'Corpus WMT24    : {len(test_docs)} documents')
print(f'Contextes RAG   : {len(rag_texts)} documents FR')
print(f'Dictionnaire    : {len(raw_dict):,} entrées')

Corpus WMT24    : 49 documents
Contextes RAG   : 10995 documents FR
Dictionnaire    : 2,515 entrées


## 3. Construction de l'index FAISS avec embeddings

In [7]:
print('Chargement du modèle d\'embeddings (multilingual-e5-base)...')
embed_model = SentenceTransformer('intfloat/multilingual-e5-base')

print('Encodage des contextes RAG (peut prendre quelques minutes)...')
# multilingual-e5 requiert le préfixe "passage: " pour les documents
rag_embeddings = embed_model.encode(
    [f'passage: {t}' for t in rag_texts],
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Index FAISS (cosine similarity via inner product sur vecteurs normalisés)
dim = rag_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(rag_embeddings.astype('float32'))

print(f'Index FAISS construit : {index.ntotal} vecteurs de dimension {dim}')


def retrieve_top_k(query_text, k=3):
    """Retourne les k contextes FR les plus proches de la requête EN."""
    query_emb = embed_model.encode(
        [f'query: {query_text}'],
        normalize_embeddings=True
    ).astype('float32')
    scores, indices = index.search(query_emb, k)
    return [rag_texts[i] for i in indices[0]], scores[0]


# Test rapide
ctx_test, scores_test = retrieve_top_k(test_docs[0]['text_en'], k=3)
print(f'\nTest retrieval (doc #0) — scores : {scores_test.round(3)}')
print(f'Contexte top-1 (extrait) : {ctx_test[0][:200]}...')

Chargement du modèle d'embeddings (multilingual-e5-base)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encodage des contextes RAG (peut prendre quelques minutes)...


Batches:   0%|          | 0/172 [00:00<?, ?it/s]

Index FAISS construit : 10995 vecteurs de dimension 768

Test retrieval (doc #0) — scores : [0.819 0.819 0.818]
Contexte top-1 (extrait) : L’objectif général de cette étude était de montrer comment utiliser les renseignements recueillis dans le cadre du Programme de la sécurité des produits de consommation (le Programme) pour identifier ...


## 4. Initialisation client Mistral

In [8]:
mistral_client = Mistral(api_key=userdata.get('MISTRAL_API_KEY'))
MODEL_NAME = 'mistral-small-latest'
print(f'Client Mistral initialisé — modèle : {MODEL_NAME}')


def translate_mistral(source_en):
    """Traduction directe sans RAG."""
    prompt = f"""You are an expert medical translator. Translate the following English medical text into French.

English text to translate:
{source_en}

Provide only the French translation, without any explanation or comment."""
    response = mistral_client.chat.complete(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=1000
    )
    return response.choices[0].message.content.strip()


def translate_with_rag(source_en, k=3):
    """Traduction avec RAG (k contextes FR récupérés via embeddings)."""
    contexts, _ = retrieve_top_k(source_en, k=k)
    context_block = '\n\n'.join([
        f'Example {i+1} of French medical text:\n{ctx}'
        for i, ctx in enumerate(contexts)
    ])
    prompt = f"""You are an expert medical translator. Translate the following English medical text into French.
To help you, here are examples of French medical texts from the same domain:

{context_block}

English text to translate:
{source_en}

Provide only the French translation, without any explanation or comment."""
    response = mistral_client.chat.complete(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=1000
    )
    return response.choices[0].message.content.strip()


# Test rapide
print('\nTest Mistral seul (doc #0)...')
print(translate_mistral(test_docs[0]['text_en'])[:200])
print('\nTest Mistral + RAG (doc #0)...')
print(translate_with_rag(test_docs[0]['text_en'])[:200])


Client Mistral initialisé — modèle : mistral-small-latest

Test Mistral seul (doc #0)...
L'objectif de ce travail était de décrire les décès par asphyxie mécanique survenus à Abidjan afin de contribuer à leur prévention. Il s'agissait d'une étude rétrospective et descriptive menée sur une

Test Mistral + RAG (doc #0)...
L’objectif de ce travail était de décrire les décès par asphyxie mécanique survenus à Abidjan afin de contribuer à leur prévention. Il s’agissait d’une étude rétrospective et descriptive menée sur une


## 5. Génération des 49 traductions — Mistral seul

In [9]:
print('=' * 70)
print('GÉNÉRATION TRADUCTIONS — Mistral seul')
print('=' * 70 + '\n')

mistral_translations = []
for i, doc in enumerate(tqdm(test_docs, desc='Mistral')):
    try:
        translation = translate_mistral(doc['text_en'])
        mistral_translations.append(translation)
        time.sleep(0.3)
    except Exception as e:
        print(f'Erreur doc {i} : {e}')
        mistral_translations.append('')

import os, json as json2
os.makedirs('results', exist_ok=True)
with open('results/mistral_translations.json', 'w', encoding='utf-8') as f:
    json2.dump(mistral_translations, f, ensure_ascii=False, indent=2)
print(f'\n{len(mistral_translations)} traductions Mistral générées et sauvegardées.')
print(f'\nExemple doc #0 :\n{mistral_translations[0][:400]}...')


GÉNÉRATION TRADUCTIONS — Mistral seul



Mistral:   0%|          | 0/49 [00:00<?, ?it/s]


49 traductions Mistral générées et sauvegardées.

Exemple doc #0 :
L'objectif de ce travail était de décrire les décès par asphyxie mécanique survenus à Abidjan afin de contribuer à leur prévention. Il s'agissait d'une étude rétrospective et descriptive menée sur une période de 19 ans (2002-2020) et portant sur les décès par asphyxie mécanique pris en charge par la Médecine légale. Les décès par asphyxie mécanique représentaient 1,23 % (756/60 984), concernaient ...


## 5b. Génération des 49 traductions — Mistral + RAG

In [10]:
print('=' * 70)
print('GÉNÉRATION TRADUCTIONS — Mistral + RAG (k=3)')
print('=' * 70 + '\n')

rag_translations = []
for i, doc in enumerate(tqdm(test_docs, desc='Mistral + RAG')):
    try:
        translation = translate_with_rag(doc['text_en'], k=3)
        rag_translations.append(translation)
        time.sleep(0.3)
    except Exception as e:
        print(f'Erreur doc {i} : {e}')
        rag_translations.append('')

with open('results/rag_translations.json', 'w', encoding='utf-8') as f:
    json2.dump(rag_translations, f, ensure_ascii=False, indent=2)
print(f'\n{len(rag_translations)} traductions RAG générées et sauvegardées.')
print(f'\nExemple doc #0 :\n{rag_translations[0][:400]}...')


GÉNÉRATION TRADUCTIONS — Mistral + RAG (k=3)



Mistral + RAG:   0%|          | 0/49 [00:00<?, ?it/s]


49 traductions RAG générées et sauvegardées.

Exemple doc #0 :
L’objectif de ce travail était de décrire les décès par asphyxie mécanique survenus à Abidjan afin de contribuer à leur prévention. Il s’agissait d’une étude rétrospective et descriptive menée sur une période de 19 ans (2002-2020) et portant sur les décès par asphyxie mécanique pris en charge par la Médecine légale. Les décès par asphyxie mécanique représentaient 1,23 % (756/60 984), concernaient ...


## 6. MEDCON-like sur Mistral seul et Mistral + RAG

In [11]:
pairs        = build_pairs(raw_dict)
automaton_en = build_automaton(pairs, 'en')
automaton_fr = build_automaton(pairs, 'fr')

print('Calcul MEDCON-like — Mistral seul...')
results_mistral = []
for i, doc in enumerate(tqdm(test_docs, desc='MEDCON Mistral')):
    r = medcon_grouped(doc['text_en'], mistral_translations[i], pairs, automaton_en, automaton_fr)
    results_mistral.append({
        'doc_id': i, 'medcon_f1': r['f1'],
        'medcon_precision': r['precision'], 'medcon_recall': r['recall'],
        'n_expected': r['n_expected'], 'matched': r['matched'],
        'missed': r['missed'], 'extra': r['extra'],
    })
df_mistral = pd.DataFrame(results_mistral)
print(f'  Precision : {df_mistral["medcon_precision"].mean():.3f}')
print(f'  Recall    : {df_mistral["medcon_recall"].mean():.3f}')
print(f'  F1        : {df_mistral["medcon_f1"].mean():.3f}')

print('\nCalcul MEDCON-like — Mistral + RAG...')
results_rag = []
for i, doc in enumerate(tqdm(test_docs, desc='MEDCON RAG')):
    r = medcon_grouped(doc['text_en'], rag_translations[i], pairs, automaton_en, automaton_fr)
    results_rag.append({
        'doc_id': i, 'medcon_f1': r['f1'],
        'medcon_precision': r['precision'], 'medcon_recall': r['recall'],
        'n_expected': r['n_expected'], 'matched': r['matched'],
        'missed': r['missed'], 'extra': r['extra'],
    })
df_rag = pd.DataFrame(results_rag)
print(f'  Precision : {df_rag["medcon_precision"].mean():.3f}')
print(f'  Recall    : {df_rag["medcon_recall"].mean():.3f}')
print(f'  F1        : {df_rag["medcon_f1"].mean():.3f}')


Calcul MEDCON-like — Mistral seul...


MEDCON Mistral:   0%|          | 0/49 [00:00<?, ?it/s]

  Precision : 0.459
  Recall    : 0.429
  F1        : 0.439

Calcul MEDCON-like — Mistral + RAG...


MEDCON RAG:   0%|          | 0/49 [00:00<?, ?it/s]

  Precision : 0.452
  Recall    : 0.429
  F1        : 0.435


## 7. BLEU sur les traductions RAG

In [15]:
## 7. BLEU sur Mistral seul et Mistral + RAG
import sacrebleu

print('Calcul BLEU — Mistral seul...')
for i, doc in enumerate(tqdm(test_docs, desc='BLEU Mistral')):
    bleu = sacrebleu.sentence_bleu(mistral_translations[i], [doc['translation_fr']]).score if mistral_translations[i] else 0.0
    df_mistral.loc[i, 'bleu'] = bleu
print(f'BLEU moyen Mistral : {df_mistral["bleu"].mean():.2f}')

print('\nCalcul BLEU — Mistral + RAG...')
for i, doc in enumerate(tqdm(test_docs, desc='BLEU RAG')):
    bleu = sacrebleu.sentence_bleu(rag_translations[i], [doc['translation_fr']]).score if rag_translations[i] else 0.0
    df_rag.loc[i, 'bleu'] = bleu
print(f'BLEU moyen RAG : {df_rag["bleu"].mean():.2f}')

Calcul BLEU — Mistral seul...


BLEU Mistral:   0%|          | 0/49 [00:00<?, ?it/s]

BLEU moyen Mistral : 48.14

Calcul BLEU — Mistral + RAG...


BLEU RAG:   0%|          | 0/49 [00:00<?, ?it/s]

BLEU moyen RAG : 49.08


print('Calcul BLEU — Mistral seul...')
for i, doc in enumerate(tqdm(test_docs, desc='BLEU Mistral')):
    bleu = sacrebleu.sentence_bleu(mistral_translations[i], [doc['translation_fr']]).score if mistral_translations[i] else 0.0
    df_mistral.loc[i, 'bleu'] = bleu
print(f'BLEU moyen Mistral : {df_mistral["bleu"].mean():.2f}')

print('\nCalcul BLEU — Mistral + RAG...')
for i, doc in enumerate(tqdm(test_docs, desc='BLEU RAG')):
    bleu = sacrebleu.sentence_bleu(rag_translations[i], [doc['translation_fr']]).score if rag_translations[i] else 0.0
    df_rag.loc[i, 'bleu'] = bleu
print(f'BLEU moyen RAG : {df_rag["bleu"].mean():.2f}')


In [16]:
print('\n' + '=' * 65)
print('COMPARAISON Mistral vs Mistral + RAG')
print('=' * 65)
print(f"\n{'Système':<25} {'MEDCON-like F1':>15} {'BLEU':>8}")
print('-' * 50)
print(f"{'Mistral':<25} {df_mistral['medcon_f1'].mean():>15.3f} {df_mistral['bleu'].mean():>8.2f}")
print(f"{'Mistral + RAG (k=3)':<25} {df_rag['medcon_f1'].mean():>15.3f} {df_rag['bleu'].mean():>8.2f}")

delta_medcon = df_rag['medcon_f1'] - df_mistral['medcon_f1']
delta_bleu   = df_rag['bleu']      - df_mistral['bleu']

print(f'\nDelta MEDCON-like F1 (RAG - Mistral) : {delta_medcon.mean():+.3f}')
print(f'  Docs où RAG > Mistral : {(delta_medcon > 0).sum()}/{len(test_docs)}')
print(f'  Docs où RAG < Mistral : {(delta_medcon < 0).sum()}/{len(test_docs)}')
print(f'  Docs égaux            : {(delta_medcon == 0).sum()}/{len(test_docs)}')
print(f'\nDelta BLEU (RAG - Mistral) : {delta_bleu.mean():+.2f}')



COMPARAISON Mistral vs Mistral + RAG

Système                    MEDCON-like F1     BLEU
--------------------------------------------------
Mistral                             0.439    48.14
Mistral + RAG (k=3)                 0.435    49.08

Delta MEDCON-like F1 (RAG - Mistral) : -0.004
  Docs où RAG > Mistral : 0/49
  Docs où RAG < Mistral : 1/49
  Docs égaux            : 48/49

Delta BLEU (RAG - Mistral) : +0.94


## 9. Exemples qualitatifs

In [17]:
best_idx  = delta_medcon.idxmax()
worst_idx = delta_medcon.idxmin()

for idx, label in [(best_idx, 'MEILLEURE AMÉLIORATION RAG'), (worst_idx, 'PLUS FORTE DÉGRADATION RAG')]:
    print(f"\n{'#'*80}\n  {label}  (doc #{idx})\n{'#'*80}")
    print(f"\nSOURCE EN :\n{test_docs[idx]['text_en'][:400]}...")
    print(f"\nMistral FR :\n{mistral_translations[idx][:400]}...")
    print(f"\nRAG FR :\n{rag_translations[idx][:400]}...")
    print(f"\n{'-'*60}")
    print(f"MEDCON-like F1  Mistral={df_mistral.iloc[idx]['medcon_f1']:.3f}  RAG={df_rag.iloc[idx]['medcon_f1']:.3f}  Delta={delta_medcon.iloc[idx]:+.3f}")
    print(f"BLEU            Mistral={df_mistral.iloc[idx]['bleu']:.1f}  RAG={df_rag.iloc[idx]['bleu']:.1f}")
    print(f"\nMatchées RAG : {df_rag.iloc[idx]['matched'][:5]}")
    print(f"Manquées RAG : {df_rag.iloc[idx]['missed'][:5]}")



################################################################################
  MEILLEURE AMÉLIORATION RAG  (doc #0)
################################################################################

SOURCE EN :
The aim of this work was to describe the deaths by mechanical as phyxiation that occurred in Abidjan in order to contribute to their prevention. This was a retrospective and descriptive study carried out over a period of 19 years (2002-2020) and relating to deaths by mechanical asphyxia treated by Forensic Medicine. Deaths by mechanical asphyxiation represented 1.23% (756/60,984), concerned men (8...

Mistral FR :
L'objectif de ce travail était de décrire les décès par asphyxie mécanique survenus à Abidjan afin de contribuer à leur prévention. Il s'agissait d'une étude rétrospective et descriptive menée sur une période de 19 ans (2002-2020) et portant sur les décès par asphyxie mécanique pris en charge par la Médecine légale. Les décès par asphyxie mécanique représentaient 1